# LogPress

### Data Loading

Load all log datasets and examine their formats.

In [10]:
# Library imports

import time
from pathlib import Path
from collections import Counter

try:
    import pandas as pd
except ImportError:
    print("Installing pandas...")
    %pip install pandas
    import pandas as pd

print("All libraries loaded.")

All libraries loaded.


In [11]:
# Dataset paths
DATASETS_DIR = Path("datasets")

DATASET_FILES = {
    "Apache": DATASETS_DIR / "Apache.log",
    "BGL": DATASETS_DIR / "BGL.log",
    "Hadoop": DATASETS_DIR / "Hadoop.log",
    "HDFS": DATASETS_DIR / "HDFS.log",
    "HealthApp": DATASETS_DIR / "HealthApp.log",
    "Linux": DATASETS_DIR / "Linux.log",
    "Mac": DATASETS_DIR / "Mac.log",
    "OpenSSH": DATASETS_DIR / "OpenSSH.log",
    "Openstack": DATASETS_DIR / "Openstack.log",
    "Spark": DATASETS_DIR / "Spark.log",
    "Proxifier": DATASETS_DIR / "Proxifier.log",
    "Zookeeper": DATASETS_DIR / "Zookeeper.log",
}

# Verify all files exist
for name, path in DATASET_FILES.items():
    if path.exists():
        print(f"{name}: {path} (EXISTS)")
    else:
        print(f"{name}: {path} (MISSING)")

Apache: datasets/Apache.log (EXISTS)
BGL: datasets/BGL.log (EXISTS)
Hadoop: datasets/Hadoop.log (EXISTS)
HDFS: datasets/HDFS.log (EXISTS)
HealthApp: datasets/HealthApp.log (EXISTS)
Linux: datasets/Linux.log (EXISTS)
Mac: datasets/Mac.log (EXISTS)
OpenSSH: datasets/OpenSSH.log (EXISTS)
Openstack: datasets/Openstack.log (EXISTS)
Spark: datasets/Spark.log (EXISTS)
Proxifier: datasets/Proxifier.log (EXISTS)
Zookeeper: datasets/Zookeeper.log (EXISTS)


In [12]:
# Functions to use in data loading
def load_logs(filepath):
    logs = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n\r")
            if line:  # skip empty lines
                logs.append(line)
    return logs


def get_file_size_mb(filepath):
    return filepath.stat().st_size / (1024 * 1024)

In [13]:
# Load all datasets
datasets = {}

print("Loading datasets...\n")
print(f"{'Dataset':<12} {'Lines':>12} {'Size (MB)':>12}")
print("-" * 40)

total_lines = 0
total_size = 0

for name, path in DATASET_FILES.items():
    start = time.time()
    logs = load_logs(path)
    elapsed = time.time() - start
    
    size_mb = get_file_size_mb(path)
    datasets[name] = logs
    
    total_lines += len(logs)
    total_size += size_mb
    
    print(f"{name:<12} {len(logs):>12,} {size_mb:>12.1f}")

print("-" * 40)
print(f"{'TOTAL':<12} {total_lines:>12,} {total_size:>12.1f}")
print("\nAll datasets loaded.")

Loading datasets...

Dataset             Lines    Size (MB)
----------------------------------------
Apache             51,978          4.7
BGL             4,747,963        708.8
Hadoop            179,993         30.4
HDFS           11,175,629       1504.9
HealthApp         253,395         22.4
Linux              25,567          2.2
Mac               117,283         16.1
OpenSSH           638,947         67.3
Openstack         189,386         53.4
Spark          16,075,118       1734.7
Proxifier          21,320          2.4
Zookeeper          74,273          9.9
----------------------------------------
TOTAL          33,550,852       4157.2

All datasets loaded.


In [14]:
# Preview first 3 logs from each dataset we loaded
print("Log Format Samples\n")

for name, logs in datasets.items():
    print(f"=== {name} ===")
    for i in range(min(3, len(logs))):
        # Truncate long lines for display
        line = logs[i]
        if len(line) > 120:
            line = line[:120] + "..."
        print(f"  {line}")
    print()

Log Format Samples

=== Apache ===
  [Thu Jun 09 06:07:04 2005] [notice] LDAP: Built with OpenLDAP LDAP SDK
  [Thu Jun 09 06:07:04 2005] [notice] LDAP: SSL support unavailable
  [Thu Jun 09 06:07:04 2005] [notice] suEXEC mechanism enabled (wrapper: /usr/sbin/suexec)

=== BGL ===
  - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.363779 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction c...
  - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.527847 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction c...
  - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.675872 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction c...

=== Hadoop ===
  2015-10-17 21:48:16,337 INFO [main] org.apache.hadoop.metrics2.impl.MetricsConfig: loaded properties from hadoop-metrics...
  2015-10-17 21:48:16,509 INFO [main] org.apache.hadoop.metrics2.impl.MetricsSystemImpl: Scheduled snapshot period at 10 s...
  2015-10-17 21:48:16,509 INFO [main] org.apache.hadoop.metrics2.impl.Metr

In [15]:
# Statistics per dataset
stats = []

for name, logs in datasets.items():
    lengths = [len(log) for log in logs]
    stats.append({
        "Dataset": name,
        "Log Count": len(logs),
        "Min Length": min(lengths),
        "Max Length": max(lengths),
        "Avg Length": round(sum(lengths) / len(lengths), 1),
        "Total Chars": sum(lengths),
    })

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

  Dataset  Log Count  Min Length  Max Length  Avg Length  Total Chars
   Apache      51978          48         187        94.7      4923706
      BGL    4747963          94         928       155.5    738436986
   Hadoop     179993          65         592       176.2     31707290
     HDFS   11175629          75         320       139.2   1555631648
HealthApp     253395          48         294        90.9     23023140
    Linux      25567          29        1030        90.9      2324119
      Mac     117283           1        1314       141.8     16634140
  OpenSSH     638947          65         188       109.4     69897118
Openstack     189386         100         450       294.7     55816574
    Spark   16075118          36        1163       112.2   1802875703
Proxifier      21320          91         224       117.2      2498071
Zookeeper      74273          75         387       138.1     10256034


### Section 2: Tokenization

Split logs into tokens. 
Tokenizer will handle those delimiters ( Spaces, Brackets [] () , Pipes | , Colons : , Equals = ).
Count Token Frequency and Distribution per dataset.

In [16]:
# Tokenization function
def tokenize(logs):
    tokens = []
    current_token = ""
    
    i = 0
    # Delimiters: space/tab, brackets, pipes, colons, equals, commas
    bracket_delims = "[](){}"
    symbol_delims = "|:=,"
    whitespace = " \t"
    
    while i < len(logs):
        char = logs[i]
        
        if char in whitespace:
            if current_token:
                tokens.append(current_token)
                current_token = ""
        
        elif char in bracket_delims or char in symbol_delims:
            if current_token:
                tokens.append(current_token)
                current_token = ""
            tokens.append(char)
        
        else:
            current_token += char
        i += 1
    
    if current_token:
        tokens.append(current_token)
    
    return tokens

In [ ]:
# Tokenize all unstructured logs
tokenized_datasets = {}

print("Tokenizing all datasets...\n")
print(f"{'Dataset':<12} {'Logs':>12} {'Total Tokens':>15} {'Avg Tokens/Log':>15}")
print("-" * 58)

for name, logs in datasets.items():
    start = time.time()
    tokenized_logs = [tokenize(log) for log in logs]
    elapsed = time.time() - start
    
    total_tokens = sum(len(t) for t in tokenized_logs)
    avg_tokens = total_tokens / len(logs) if logs else 0
    
    tokenized_datasets[name] = tokenized_logs
    
    print(f"{name:<12} {len(logs):>12,} {total_tokens:>15,} {avg_tokens:>15.1f}")

print("\nTokenization complete.")

Tokenizing all datasets...

Dataset              Logs    Total Tokens  Avg Tokens/Log
----------------------------------------------------------
Apache             51,978       1,205,915            23.2
BGL             4,747,963      96,104,571            20.2
Hadoop            179,993       4,584,261            25.5
HDFS           11,175,629     174,524,695            15.6
HealthApp         253,395       4,277,624            16.9
Linux              25,567         612,978            24.0
Mac               117,283       3,059,934            26.1
OpenSSH           638,947      17,821,657            27.9
Openstack         189,386       6,462,675            34.1


In [ ]:
# Token frequency

all_tokens = []
for name, tokenized_logs in tokenized_datasets.items():
    for tokens in tokenized_logs:
        all_tokens.extend(tokens)

token_counts = Counter(all_tokens)

print("Top 20 Most Common Tokens (across all datasets):\n")
print(f"{'Rank':<6} {'Token':<30} {'Count':>15}")
print("-" * 55)

for rank, (token, count) in enumerate(token_counts.most_common(20), 1):
    display_token = token if len(token) <= 28 else token[:25] + "..."
    print(f"{rank:<6} {display_token:<30} {count:>15,}")